In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

UsageError: Line magic function `%` not found.


In [ ]:
# Load Data
try:
    df = pd.read_csv("data/results.csv")
    df = df[df["algorithm"] != "relative_neighborhood"]
    print(f"Loaded {len(df)} rows from compare_results.csv")
    display(df.head())
except FileNotFoundError:
    print("No compare_results.csv found.")


In [ ]:
# Filter data for inv_knn+probabilistic algorithm
alg_name = 'inv_knn+n_probabilistic'  # or just contains probabilistic

subset = df[df['algorithm'].str.contains('probabilistic', na=False) & df['algorithm'].str.contains('inv_knn', na=False)]

# Group by K and player
grouped = subset.groupby(['k_nn', 'player'])['pth_len'].agg(['mean', 'sem']).unstack()

# Extract data
if 'human' in grouped.columns.levels[1] and 'greedy' in grouped.columns.levels[1]:
    human_data = grouped['mean', 'human'].dropna()
    human_errors = grouped['sem', 'human'].dropna()
    
    comp_data = grouped['mean', 'greedy'].dropna()
    comp_errors = grouped['sem', 'greedy'].dropna()
    
    # Use intersection of index in case one has missing K
    common_idx = human_data.index.intersection(comp_data.index)
    human_data = human_data.loc[common_idx]
    human_errors = human_errors.loc[common_idx]
    comp_data = comp_data.loc[common_idx]
    comp_errors = comp_errors.loc[common_idx]

    k_values = common_idx 
    x = np.arange(len(k_values))  # Base numeric locations for the groups [0, 1, 2...]
    width = 0.35  # The width of the bars
    
    # 4. Create the Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Plot Human bars (shifted left)
    rects_human = ax.bar(x - width/2, 
                         human_data.values, 
                         width, 
                         yerr=human_errors.values, 
                         capsize=5, 
                         label='Human', 
                         color='teal', 
                         edgecolor='black')
    
    # Plot Computer bars (shifted right)
    rects_comp = ax.bar(x + width/2, 
                        comp_data.values, 
                        width, 
                        yerr=comp_errors.values, 
                        capsize=5, 
                        label='Computer', 
                        color='coral', 
                        edgecolor='black')
    
    # 5. Add Labels, Title, and Styling
    ax.set_title('Human vs. Computer Performance by Graph Connectivity (probabilistic N)', fontsize=14)
    ax.set_xlabel('K', fontsize=12)
    ax.set_ylabel('Avg. Path Length', fontsize=12)
    
    # Set the x-ticks to be in the middle of the groups, and label them with 'k' values
    ax.set_xticks(x)
    ax.set_xticklabels(k_values)
    
    # Add a grid (horizontal only is usually best for bar charts)
    ax.grid(True, axis='y', linestyle='--', alpha=0.6)
    
    # Add Legend
    ax.legend()
    
    # 6. Add Data Annotations (Replaces your plt.text loop)
    # Note: padding controls how far above the error bar the text sits
    ax.bar_label(rects_human, padding=10, fmt='%.2f', fontweight='bold', size=9)
    ax.bar_label(rects_comp, padding=10, fmt='%.2f', fontweight='bold', size=9)
    
    plt.tight_layout()
    plt.show()
else:
    print(f"Missing player data in subset. Found players: {subset['player'].unique() if not subset.empty else 'None'}")
